In [ ]:
# W8 Day4 - Runtime 无状态执行
# matplotlib 中文字体配置
from matplotlib import font_manager
import matplotlib.pyplot as plt
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print(f"中文字体配置完成: {font_name}")

# 🎯 今日学习目标 | 第8周-Day4：Runtime 无状态执行

> **为什么 Runtime 不保存状态？**
> Runtime 是"手术室"不是"病房"——所有信息通过 FrozenExecutionContext 带进来，结果通过 ExecutionResult 带出去，手术室本身不保存任何病人数据。

## 📅 学习进度

```
W1-W7  ████████████████████ ✅ AI基础（已完成）
W8     ████████████░░░░░░░ 🔥 LangChat End-to-End Journey (Day4/7)
  D1   ✅ 用户意图：Agent Host 怎么连进来？
  D2   ✅ ApplicationContract：业务契约不是 API 文档
  D3   ✅ Blueprint → Compiler → ExecutionPlanIR：意图变成可执行的
  D4   🔥 Runtime：执行计划怎么跑起来的？
  D5   ⬜ Capability + Connector：怎么连到业务系统？
```

**进度: Week8 Day4/7 | 总体 33/126**

# 🔄 往期回顾

| Day | 主题 | 核心认知 |
|-----|------|----------|
| D1 | 用户意图 | LangChat 是企业能力平台，Agent Host 直接调用 |
| D2 | ApplicationContract | Contract 是业务治理一等对象，不可变版本 |
| D3 | Blueprint→Compiler→IR | Blueprint 是制品，Compiler 确定性翻译 |

### 今日关联
- **D2 ApplicationContract** → `effect_policy` / `required_scopes` 进入 FrozenExecutionContext
- **D3 ExecutionPlanIR** → 打包进 SkillRelease v2 manifest，Runtime 只装载 SkillRelease
- **D1 Agent Host** → 不直接调 Runtime；请求经 Gateway 构造 FEC 后进入

# 📚 Part 1：为什么 Runtime 不保存状态？

### 通俗类比：手术室 vs 病房

- **病房（有状态）**：病人住进来，医生每天查房，病历堆在床头柜
- **手术室（无状态）**：病人推入时带所有术前信息，手术结束带走所有结果

如果手术室存了上一个病人的信息 → 下一个病人可能拿到错药。

| 需求 | 有状态风险 | 无状态优势 |
|------|----------|----------|
| 水平扩展 | 必须 sticky session | 任意实例处理任意请求 |
| 安全性 | 上下文可能泄漏 | 每次执行完全隔离 |
| 可审计 | 内存状态无法复现 | FEC 不可变，完整审计 |
| 故障恢复 | 实例崩溃丢状态 | 无状态 = 无丢失 |

# 📚 Part 2：FrozenExecutionContext 九组字段

ADR-007 §7 冻结了 FEC 的完整 wire JSON schema。所有执行所需信息在执行前一次性"冻结"成不可变对象。

| 字段组 | 职责 | ADR 来源 |
|--------|------|----------|
| `identity` | 谁？租户、工作空间、调用者身份、委托链 | ADR-002 D1 |
| `policy` | 能做什么？效果策略快照、调用深度限制 | Charter §6.1 |
| `contract_route` | 做哪份契约？精确 digest + Deployment | ADR-005 D-1 |
| `artifact_digests` | 用哪些制品？SkillRelease digest、RuntimeABI | ADR-007 D-1/D-2 |
| `knowledge_capability` | 需要哪些知识和能力？ | Domain Model |
| `trace_audit` | 怎么追踪？request_id、trace_id | ADR-004 |
| `timing_order` | 什么时候？冻结时间、执行序号 | AS §14.1 |
| `execution_boundary` | 什么时候停？最大时长、最大成本 | AS §14.1 |
| `audit` | 怎么审计？完整性证明、构建者身份 | AS §14.2 |

### 三大不可协商原则

| 原则 | 含义 | 硬约束 |
|------|------|--------|
| **无状态** | 不保存跨请求状态 | HC-1 |
| **FEC 不可变** | 创建后不可修改 | HC-1 |
| **封闭性** | 零 workflow import | ADR-005 D-5 |

In [ ]:
# FrozenExecutionContext 九组字段可视化：手术室信息流
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

fig, ax = plt.subplots(1, 1, figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('FrozenExecutionContext 数据流：手术室模型', fontsize=16, fontweight='bold', pad=20)

# FEC 盒子
fec_box = patches.FancyBboxPatch((4.5, 3), 5, 5, boxstyle="round,pad=0.3",
                                   facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2)
ax.add_patch(fec_box)
ax.text(7, 7.7, 'FrozenExecutionContext（不可变）', ha='center', fontsize=13,
        fontweight='bold', color='#1565C0')

# 九组字段
fields = [
    ('identity', '谁？'), ('policy', '做什么？'), ('contract_route', '哪份契约？'),
    ('artifact_digests', '哪些制品？'), ('knowledge_capability', '哪些能力？'),
    ('trace_audit', '怎么追踪？'), ('timing_order', '什么时候？'),
    ('execution_boundary', '什么时候停？'), ('audit', '怎么审计？')
]
colors = ['#BBDEFB', '#C8E6C9', '#FFF9C4', '#FFE0B2', '#E1BEE7',
          '#B2EBF2', '#F8BBD0', '#DCEDC8', '#FFCCBC']
for i, (name, desc) in enumerate(fields):
    row = i // 3
    col = i % 3
    x = 5.0 + col * 1.6
    y = 7.0 - row * 1.2
    box = patches.FancyBboxPatch((x-0.65, y-0.4), 1.4, 0.8,
                                  boxstyle="round,pad=0.1",
                                  facecolor=colors[i], edgecolor='gray')
    ax.add_patch(box)
    ax.text(x, y+0.15, name, ha='center', va='center', fontsize=7, fontweight='bold')
    ax.text(x, y-0.15, desc, ha='center', va='center', fontsize=6, color='gray')

# 左侧输入
ax.annotate('Agent Host\n请求', xy=(4.5, 5.5), fontsize=11, ha='center', va='center',
            fontweight='bold', color='#D32F2F',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFCDD2', edgecolor='#D32F2F'))
ax.annotate('', xy=(4.5, 5.5), xytext=(2.5, 5.5),
            arrowprops=dict(arrowstyle='->', color='#D32F2F', lw=2))

# 中间 Gateway
ax.annotate('Gateway\n构造 FEC', xy=(2.5, 5.5), fontsize=9, ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='#FFF9C4', edgecolor='#F9A825'))

# 右侧输出
ax.annotate('ExecutionResult\n七字段输出', xy=(11.5, 5.5), fontsize=11, ha='center', va='center',
            fontweight='bold', color='#388E3C',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#C8E6C9', edgecolor='#388E3C'))
ax.annotate('', xy=(11.5, 5.5), xytext=(9.5, 5.5),
            arrowprops=dict(arrowstyle='->', color='#388E3C', lw=2))

# Runtime
rt_box = patches.FancyBboxPatch((9.8, 4.2), 1.5, 2.6, boxstyle="round,pad=0.2",
                                  facecolor='#E8F5E9', edgecolor='#388E3C', linewidth=2)
ax.add_patch(rt_box)
ax.text(10.55, 6.4, 'Runtime', ha='center', fontsize=10, fontweight='bold', color='#388E3C')
ax.text(10.55, 5.7, 'execute()', ha='center', fontsize=9, fontstyle='italic')
ax.text(10.55, 5.1, '只读取 FEC\n永不修改', ha='center', fontsize=7, color='gray')

# 底部 DeploymentRevision
ax.annotate('DeploymentRevision\n(SkillRelease 闭包)', xy=(7, 1.5), fontsize=9, ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='#E0E0E0', edgecolor='#616161'))
ax.annotate('', xy=(7, 3), xytext=(7, 2.3),
            arrowprops=dict(arrowstyle='->', color='#616161', lw=1.5))

# 无状态标注
ax.text(7, 0.5, '⚠️ Runtime 零状态：不保存任何跨请求数据', ha='center',
        fontsize=10, color='#E65100', fontstyle='italic')

plt.tight_layout()
plt.savefig('w8d4_fec_flow.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ FrozenExecutionContext 数据流图已生成')

# 📚 Part 3：当前代码实现

### 3.1 Runtime 包结构

```
langchat/runtime/
├── canonical_entry.py      # execute() 入口
├── frozen_execution_context.py  # FEC 不可变 dataclass
├── loader.py               # RuntimeLoader（WP-05 stub）
├── types.py                # 类型别名（Callable）
├── deployment_revision.py   # DeploymentRevision 闭包
└── production.py            # ProductionFrozenExecutionContext
```

### 3.2 execute() 签名

```python
execute(deployment_revision, frozen_context, input,
        *, runtime_factory, kb_search_fn, llm_chat_fn, tool_call_fn)
```

四个 keyword-only 参数无默认值 → 封闭性保证。

### 3.3 七字段 Fallback

`execute()` 永不抛异常，失败返回：
```python
keys = ("summary", "details", "references", "assumptions",
        "human_review_required", "next_actions", "risk_flags")
```

In [ ]:
# 模拟 FrozenExecutionContext 不可变性
from dataclasses import dataclass
from typing import Mapping

@dataclass(frozen=True)
class SimulatedFEC:
    frozen_context_id: str
    subject_closure_digest: str
    policy_floor_digest: str
    trace_id: str
    policy_snapshot: Mapping[str, object]

fec = SimulatedFEC(
    frozen_context_id="fec-prod-1",
    subject_closure_digest="sha256:" + "a" * 64,
    policy_floor_digest="sha256:" + "p" * 64,
    trace_id="trace-001",
    policy_snapshot={"effect_policy": "read_only", "call_chain_depth_limit": 8}
)

print(f"FEC 创建成功: context_id={fec.frozen_context_id}")
print(f"策略快照: {fec.policy_snapshot}")

# 尝试修改 → 应该抛出 FrozenInstanceError
try:
    fec.trace_id = "mutated"
    print("❌ FEC 被修改了！这是安全漏洞！")
except AttributeError as e:
    print(f"✅ FEC 不可变：{e}")

In [ ]:
# Gap Analysis 可视化：目标态 vs 当前实现
import matplotlib.pyplot as plt
import numpy as np

dimensions = ['RuntimeLoader', 'FEC wire 完整性', 'Signature 验证',
              'Compatibility Matrix', 'AIBOM', '封闭性',
              'execute() 永不抛异常', 'FEC 不可变']
target = [100, 100, 100, 100, 100, 100, 100, 100]
current = [10, 40, 0, 30, 0, 100, 100, 100]

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(dimensions))
width = 0.35

bars1 = ax.bar(x - width/2, target, width, label='目标态（ADR-007）',
               color='#1565C0', alpha=0.8)
bars2 = ax.bar(x + width/2, current, width, label='当前实现（WP-05/07）',
               color=['#E53935' if c < 50 else '#FFA726' if c < 80 else '#43A047' for c in current])

ax.set_ylabel('完成度 (%)', fontsize=12)
ax.set_title('Runtime Gap Analysis：目标态 vs 当前实现', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(dimensions, rotation=30, ha='right', fontsize=9)
ax.legend()
ax.set_ylim(0, 115)

# 标注数值
for bar in bars2:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 2,
            f'{int(height)}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('w8d4_gap_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gap Analysis 图已生成')

# 📚 Part 4：Gap Analysis

| 维度 | 目标态 | 当前代码 | Gap |
|------|--------|----------|-----|
| RuntimeLoader | OCI pull + layer 验证 + Compat Matrix | Stub：直接返回对象 | 🔴 |
| FEC wire | 完整 9 组字段 | 简化 profile | 🟡 |
| Signature | Sigstore cosign 两点验签 | 未实现 | 🔴 |
| Compat Matrix | semver + 366 天弃用窗口 | 基础注册存在 | 🟡 |
| 封闭性 | 零 workflow import | ✅ 已验证 | 🟢 |
| execute() 永不抛异常 | 七字段 fallback | ✅ 已实现 | 🟢 |
| FEC 不可变 | frozen=True | ✅ 已实现 | 🟢 |

**最大 Gap：RuntimeLoader。** 当前 stub 下安全链路没有闭合。

# 📚 Part 5：今天多理解了什么？

| 以前以为 | 现在知道 |
|---------|--------|
| Runtime 就是"跑代码的引擎" | Runtime 是"手术室"——封闭、无状态、所有信息通过 FEC 传入 |
| 无状态就是"不存数据库" | 无状态 = 所有执行信息显式传入 FEC，可审计、可复现 |
| Runtime 直接加载 Blueprint 执行 | Runtime 只装载 SkillRelease（通过 DeploymentRevision 闭包） |
| 异常应该直接抛给调用者 | execute() 永不抛异常，返回七字段结构化 fallback |

### 🔮 重新设计还会这样做吗？

**三大原则不可协商**：无状态（水平扩展）、FEC 不可变（审计基础）、封闭性（可替换性）。

**可能改进**：FEC 九组字段可能过大，可考虑拆分核心和扩展 FEC。

# 📚 Part 6：课堂练习

### 练习 1：画 FrozenExecutionContext 数据流图
画出 Agent Host → Gateway（构造 FEC）→ Runtime（execute()）→ ExecutionResult 的完整路径。

### 练习 2：对比有状态 vs 无状态 Runtime
假设 Runtime 保存了上一次执行结果，列出可能出现的安全问题。

### 练习 3：阅读真实代码
- `runtime/canonical_entry.py` → execute() 入口
- `runtime/frozen_execution_context.py` → FEC dataclass
- `runtime/loader.py` → RuntimeLoader stub

# 📚 Part 7：课后测试

**Q1. LangChat Runtime 为什么不保存状态？**
A) 节省内存  B) 支持水平扩展、安全隔离和确定性执行  C) 当前版本没实现  D) 有状态影响 LLM 质量

**Q2. FrozenExecutionContext 有哪些 profile？**
A) production/staging  B) runtime_application/operation  C) online/offline  D) sync/async

**Q3. execute() 为什么永不抛异常？**
A) 代码没 bug  B) 所有失败返回七字段 fallback  C) 异常机制被禁用  D) 不处理失败

**Q4. RuntimeLoader 当前是什么状态？**
A) 完整实现  B) Stub：直接返回 DeploymentRevision  C) 不存在  D) 完整但未测试

**Q5. 封闭性（零 Workflow import）意味着什么？**
A) 不能执行工作流  B) 通过参数注入执行框架  C) 必须换语言  D) WorkflowSpec 已删除

> ✅ 答案：B B B B B

# 📖 术语表

| 英文 | 音标 | 中文 |
|------|------|------|
| Runtime | /ˈraɪntaɪm/ | 运行时：执行 ExecutionPlanIR 的环境 |
| FrozenExecutionContext | /ˈfroʊzən ɪkˈsɛkjuːʃən ˈkɒntɛkst/ | 冻结执行上下文 |
| DeploymentRevision | /dɪˈplɔɪmənt rɪˈvɪʒən/ | 部署修订：完整运行时闭包 |
| RuntimeABI | /ˈraɪntaɪm eɪ biː aɪ/ | 运行时应用二进制接口 |
| Compatibility Matrix | /kəmˌpætəˈbɪləti ˈmeɪtrɪks/ | 兼容性矩阵 |
| Fallback | /ˈfɔːlbæk/ | 降级回退 |
| Hermetic | /hɜːrˈmɛtɪk/ | 封闭的 |
| SkillRelease | /skɪl rɪˈliːs/ | 能力发布制品 |
| Signature | /ˈsɪɡnətʃə/ | 签名 |
| execute() | /ɪkˈsækjuːt/ | 执行入口 |

# 📚 真实参考

| 文档 | 路径 |
|------|------|
| ADR-007 | `langchat-docs/.../ADR-007-RuntimeABI-CompatMatrix-FrozenExecutionContext-wire.md` |
| ADR-005 | `langchat-docs/.../ADR-005-Blueprint-artifact-chain-and-ApplicationContract.md` |
| canonical_entry.py | `langchat/apps/backend/langchat/runtime/canonical_entry.py` |
| frozen_execution_context.py | `langchat/apps/backend/langchat/runtime/frozen_execution_context.py` |
| loader.py | `langchat/apps/backend/langchat/runtime/loader.py` |
| test_runtime_hermetic_api.py | `langchat/apps/backend/tests/unit_tests/` |

# 📝 Daily Engineering Log

| 类别 | 内容 |
|------|------|
| 新增 | 理解 FEC 九组字段结构 |
| 确认 | execute() 永不抛异常；封闭性已验证 |
| 遗留 | RuntimeLoader 真实 OCI pull + Signature 验证未实现 |
| 技术债 | FEC wire JSON 目前只有简化 profile |
| 下一步 | Day5：Capability + Connector → Enterprise System |